# SOURCE EXPLORATION

## Setup

In [1]:
import os
import requests
import duckdb
import pandas as pd
import yfinance as yf
from dotenv import load_dotenv
from edgar import *
from fredapi import Fred
from tqdm.auto import tqdm

set_identity("John Doe john.doe@company.com")

In [2]:
"""
There is also a google cloud version.
"""

PATH_MAIN = os.path.join(os.getcwd(), 'data')
PATH_HOLDINGS = f"{PATH_MAIN}/holdings"
PATH_FINANCIALS = f"{PATH_MAIN}/financials"
PATH_TECHNICALS = f"{PATH_MAIN}/prices"
os.makedirs(PATH_MAIN, exist_ok=True)
os.makedirs(PATH_HOLDINGS, exist_ok=True)
os.makedirs(PATH_FINANCIALS, exist_ok=True)
os.makedirs(PATH_TECHNICALS, exist_ok=True)

In [3]:
ETFS = [
    'XLE', 'XLC', 'XLB', 'XLRE', 'XLU', 'XHB', 'XES', 'XLP', 'XME', 'XTL',
    'XTN', 'XLY', 'XAR', 'XSD', 'XOP', 'KIE', 'XLV', 'XHS', 'XPH', 'KCE',
    'XHE', 'XLK', 'XRT', 'XLF', 'XLI', 'KBE', 'XSW', 'XBI', 'KRE'
]

## Download holdings

In [4]:
def download_holdings(tickers, path=PATH_HOLDINGS):
    """
    Url is the official link to SSGA holdings.
    Partially cleaned to remove boiler plate cells.
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    
    for etf in tqdm(tickers):
        
        try:
            ssga_url = f'https://www.ssga.com/us/en/intermediary/library-content/products/fund-data/etfs/us/holdings-daily-us-en-{etf.lower()}.xlsx'
            response = requests.get(ssga_url, headers=headers)
            response.raise_for_status()
            
            duckdb.sql(f"""
                copy (
                    select
                        '{etf}' as etf
                        , Name as name
                        , Ticker as ticker
                        , Weight::numeric as weight
                    from read_xlsx('{ssga_url}', range = 'A5:H')
                    where true
                        and SEDOL != '-'
                        and ticker != '-'
                ) to '{path}/{etf.lower()}_holdings.parquet'
            """)
            
        except Exception as e:
            print(f"{etf} error: {e}")
            continue

In [5]:
# download_holdings(tickers=ETFS)

In [6]:
holdings = duckdb.sql(f"""
    select * 
    from read_parquet('{PATH_HOLDINGS}/*_holdings.parquet', union_by_name=True)
    order by weight desc
""").df()

holdings_by_etf = duckdb.sql(f"""
    select etf, count(*) as count, sum(weight) as weight
    from read_parquet('{PATH_HOLDINGS}/*_holdings.parquet', union_by_name=True)
    group by etf
    order by count desc
""").df()

In [7]:
tickers = holdings.ticker
unique_tickers = list(tickers.unique())
unique_tickers = unique_tickers

print(f'ETFs               {len(holdings.etf.unique())}')
print(f"ETF Holdings       {len(tickers):,.0f}")
print(f"Unique tickers     {len(unique_tickers):,.0f}")

ETFs               29
ETF Holdings       1,767
Unique tickers     1,443


## Download metadata

In [8]:
def download_metadata(etf):
    """
    Changed approach to per etf list downloads, instead of one aggreggate.
    This will help down the line, addressing difference in reporting noticeable between industries. 
    Uses edgartools, instead of official edgar api since the latter does not have this data easily-available.
    """
    tickers = holdings[(holdings.etf == etf.upper())].ticker
    metadata = []
    metadata_error = []

    print(f'Downloading {etf.upper()} metadata:')
    for t in tqdm(tickers):
        
        try:
            company = Company(t)
            metadata.append({
                'etf': etf,
                'ticker': t,
                'cik': f"CIK{str(company.cik).zfill(10)}",
                'name': company.name,
                'industry': company.industry,
                'sic': company.sic,
                'fiscal_year': company.fiscal_year_end
            })
    
        except Exception as e:
            metadata_error.append({
                'etf': etf,
                'ticker': t,
                'error': str(e)
            })
    
    metadata = pd.DataFrame(metadata)
    metadata_error = pd.DataFrame(metadata_error)
    
    metadata.to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_holdings_metadata.parquet', index=False)
    
    if not metadata_error.empty:
        metadata_error.to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_holdings_metadata_error.parquet', index=False)

In [9]:
# for e in tqdm(ETFS):
#     download_metadata(etf=e)

## Download concepts

In [10]:
def download_concepts(etf):
    """
    Similar to metadata, edgartools is the superior method to acquire 'concepts' (keys needed for the official api).
    Company reporting changes per industry, so a list of the common industry specific concepts need to be gathered,
        and iterated to acquire most, if not all, of the comparable data per industry.
    Easier way is to just get data using pre-determined concept keys, but that will misrepresent a lot of industries.
        As example, banks have interest income/expense as major revenue source, which for every other industry is a minor item.
        Utilities, being usually government contracted, have different revenue tags compared to a typical commercial industry.
    Edgartools somehow does a good job with this. Accordingly, their tags are based from the actual filings,
        which makes it almost raw data but with added help in processing the files.
    I only want a few items for this study, but this function makes it easy to get other keys
        should I decide to drill down on a specific sector.
    """
    tickers = holdings[(holdings.etf == etf.upper())].ticker
    shares_concepts = []
    revenue_concepts = []
    incomeloss_concepts = []
    shares_concepts_error = []
    revenue_concepts_error = []
    incomeloss_concepts_error = []
    check_availability = []

    print(f'Downloading {etf.upper()} concepts:')
    for t in tqdm(tickers):

        try:
            company = Company(t)
            financials = company.get_financials()
            income_stmnt = financials.income_statement()
            income_stmnt = income_stmnt.to_dataframe()

        except Exception as e:
            check_availability.append({
                'tag': 'available', 
                'ticker': t, 'error': str(e)
            })
            continue

        try:
            shares = income_stmnt[income_stmnt.standard_concept == 'SharesAverage'].concept.iloc[0]
            shares_concepts.append({
                'tag': 'shares',
                'ticker': t,
                'concept': str(shares)
            })
            
        except Exception as e:
            shares_concepts_error.append({
                'tag': 'shares',
                'ticker': t,
                'error': str(e)
            })

        try:
            revenue = income_stmnt[income_stmnt.standard_concept == 'Revenue'].concept.iloc[0]
            revenue_concepts.append({
                'tag': 'revenue',
                'ticker': t,
                'concept': str(revenue)
            })
            
        except Exception as e:
            revenue_concepts_error.append({
                'tag': 'revenue',
                'ticker': t, 
                'error': str(e)
            })

        try:
            incomeloss = income_stmnt[income_stmnt.standard_concept == 'NetIncome'].concept.iloc[0]
            incomeloss_concepts.append({
                'tag': 'incomeloss',
                'ticker': t,
                'concept': str(incomeloss)
            })
            
        except Exception as e:
            incomeloss_concepts_error.append({
                'tag': 'incomeloss',
                'ticker': t,
                'error': str(e)
            })

    pd.DataFrame(shares_concepts).to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_concepts_shares.parquet', index=False)
    pd.DataFrame(revenue_concepts).to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_concepts_revenue.parquet', index=False)
    pd.DataFrame(incomeloss_concepts).to_parquet(f'{PATH_HOLDINGS}/{etf.lower()}_concepts_incomeloss.parquet', index=False)

    if check_availability:
        pd.DataFrame(check_availability).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_check_availability.parquet', index=False)
    
    if shares_concepts_error:
        pd.DataFrame(shares_concepts_error).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_shares_missing.parquet', index=False)
    
    if revenue_concepts_error:
        pd.DataFrame(revenue_concepts_error).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_revenue_missing.parquet', index=False)
    
    if incomeloss_concepts_error:        
        pd.DataFrame(incomeloss_concepts_error).to_parquet(f'{PATH_HOLDINGS}/.{etf.lower()}_concepts_incomeloss_missing.parquet', index=False)

### XLE

In [11]:
# for e in tqdm(ETFS):
#     download_concepts(etf=e)

# download_concepts(etf='xle')

#### Missing concepts

In [12]:
xle_missing_concepts = duckdb.sql(f"""select * from read_parquet('{PATH_HOLDINGS}/.xle_concepts*', union_by_name=True)""").df()
# xle_missing_concepts

#### Fix missing OXY incomeloss concept using "us-gaap_ProfitLoss"

In [13]:
oxy = Company('OXY')
oxy = oxy.get_financials().income_statement()
oxy = oxy.to_dataframe()
oxy[oxy.concept.notna()]

,concept,label,standard_concept,2025-12-31 (FY),2024-12-31 (FY),2023-12-31 (FY),level,abstract,dimension,is_breakdown,dimension_axis,dimension_member,dimension_member_label,dimension_label,balance,weight,preferred_sign,parent_concept,parent_abstract_concept
0,oxy_RevenuesAndOtherIncomeAbstract,REVENUES AND OTHER INCOME,NaN,NaN,NaN,NaN,1,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_IncomeStatementAbstract
1,us-gaap_Revenues,Net sales,Revenue,2.159300e+10,2.201900e+10,2.315600e+10,2,False,False,False,NaN,NaN,NaN,NaN,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
2,us-gaap_Revenues,Related Party,NaN,1.280000e+08,1.460000e+08,1.530000e+08,3,False,True,False,us-gaap:RelatedPartyTransactionsByRelatedParty...,us-gaap_RelatedPartyMember,Related Party,us-gaap:RelatedPartyTransactionsByRelatedParty...,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
3,us-gaap_Revenues,Other Operating Segments And Intersegment Elim...,NaN,2.090200e+10,2.170500e+10,2.128400e+10,3,False,True,True,srt:ConsolidationItemsAxis,oxy_OtherOperatingSegmentsAndIntersegmentElimi...,Other Operating Segments And Intersegment Elim...,srt:ConsolidationItemsAxis: Other Operating Se...,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
4,us-gaap_Revenues,Other Operating Segments And Intersegment Elim...,NaN,1.279000e+09,8.860000e+08,2.433000e+09,3,False,True,True,srt:ConsolidationItemsAxis,oxy_OtherOperatingSegmentsAndIntersegmentElimi...,Other Operating Segments And Intersegment Elim...,srt:ConsolidationItemsAxis: Other Operating Se...,credit,1.0,1.0,oxy_RevenuesOperatingAndNonoperating,oxy_RevenuesAndOtherIncomeAbstract
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,us-gaap_EarningsPerShareBasic,Net income attributable to common stockholders...,NaN,1.650000e+00,2.590000e+00,4.220000e+00,2,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,us-gaap_EarningsPerShareBasicAbstract
64,us-gaap_EarningsPerShareDilutedAbstract,"PER COMMON SHARE, DILUTED",NaN,NaN,NaN,NaN,1,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,us-gaap_IncomeStatementAbstract
65,us-gaap_IncomeLossFromContinuingOperationsPerD...,Income from continuing operations—diluted (in ...,NaN,1.350000e+00,2.230000e+00,2.490000e+00,3,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,us-gaap_EarningsPerShareDiluted,us-gaap_EarningsPerShareDilutedAbstract
66,us-gaap_DiscontinuedOperationIncomeLossFromDis...,Discontinued operations—diluted (in dollars pe...,NaN,2.600000e-01,2.100000e-01,1.410000e+00,3,False,False,False,NaN,NaN,NaN,NaN,NaN,1.0,1.0,us-gaap_EarningsPerShareDiluted,us-gaap_EarningsPerShareDilutedAbstract


#### Fix missing shares concepts with ("us-gaap_EarningsPerShareBasic" / incomeloss)

In [14]:
eps_proxy = []

for t in tqdm(xle_missing_concepts.ticker):
    
    try:
        company = Company(t)
        income_stmnt = company.get_financials().income_statement().to_dataframe()
        eps = income_stmnt[income_stmnt.concept == 'us-gaap_EarningsPerShareBasic'].concept.iloc[0]
    
    except Exception as e:
        eps_proxy.append({
            'ticker': t,
            'error': str(e)
        })
        continue

if eps_proxy:
    print(f'Still mising {len(eps_proxy)}')

  0%|          | 0/6 [00:00<?, ?it/s]

In [15]:
t = 'CVX'
c = 'CIK0000093410'
url = f"https://data.sec.gov/api/xbrl/companyfacts/{c}.json"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
response.raise_for_status()
company_facts = response.json()

try:
    eps = company_facts["facts"]["us-gaap"]["EarningsPerShareBasic"]["units"]["USD/shares"]
    df_eps = pd.DataFrame(eps)
    duckdb.sql(f"""copy (
        select '{t}' as ticker, 'eps' as item, *
        from df_eps) to '{PATH_FINANCIALS}/xle/{t.lower()}_eps.parquet'
    """)
except Exception as e:
    print(e)

# DATA INGESTION

## Download macro indicators

In [16]:
load_dotenv()
fred_api_key = os.getenv('fred')
fred = Fred(api_key=fred_api_key)

# CBOE Volatility Index: VIX (VIXCLS)
vix = fred.get_series('VIXCLS')
vix.name = 'vix'
# ICE BofA AAA US Corporate Index Option-Adjusted Spread (BAMLC0A1CAAA)
aaa = fred.get_series('BAMLC0A1CAAA')
aaa.name = 'aaa'
# ICE BofA BBB US Corporate Index Option-Adjusted Spread (BAMLC0A4CBBB)
bbb = fred.get_series('BAMLC0A4CBBB')
bbb.name = 'bbb'

In [17]:
sentiment_proxies = pd.concat([vix, aaa, bbb], axis=1, sort=False).reset_index().to_parquet(f"{PATH_MAIN}/sentiment.parquet")

In [18]:
sentiment = duckdb.sql(f"""
    select
        index::date as date
        , vix, aaa, bbb
    from read_parquet("{PATH_MAIN}/sentiment.parquet")
""").df()

In [19]:
sentiment.info()

<class 'pandas.DataFrame'>
RangeIndex: 9545 entries, 0 to 9544
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    9545 non-null   datetime64[us]
 1   vix     9231 non-null   float64       
 2   aaa     787 non-null    float64       
 3   bbb     787 non-null    float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 298.4 KB


In [20]:
# sentiment.sort_values('date', ascending=False).head()

## Download financial data

In [21]:
def download_financials(tickers, revenue_concepts, incomeloss_concepts, shares_concepts, path):
    """
    This is where the official edgar api proves superior to edgartools -- historical data batches.
    A lot of inputs since the design is to have it iterate across the common concept keys, in order of frequency.
    Downloads the full json file once, before conducting per concept list matching.
    """
    os.makedirs(path, exist_ok=True)

    for _, row in tqdm(tickers[["ticker", "cik"]].iterrows()):
        t = row["ticker"]
        c = row["cik"]

        url = f"https://data.sec.gov/api/xbrl/companyfacts/{c}.json"
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        company_facts = response.json()

        for r in revenue_concepts:
            try:
                revenues = company_facts["facts"]["us-gaap"][r]["units"]["USD"]
                df_revenues = pd.DataFrame(revenues)
                duckdb.sql(f"""copy (
                    select '{t}' as ticker, 'revenues' as item, *
                    from df_revenues) to '{path}/{t.lower()}_revenues.parquet'
                """)
                break
            except Exception as e:
                continue

        for i in incomeloss_concepts:
            try:
                income = company_facts["facts"]["us-gaap"][i]["units"]["USD"]
                df_income = pd.DataFrame(income)
                duckdb.sql(f"""copy (
                    select '{t}' as ticker, 'income' as item, *
                    from df_income) to '{path}/{t.lower()}_income.parquet'
                """)
                break
            except Exception as e:
                continue

        for s in shares_concepts:
            try:
                shares = company_facts["facts"]["us-gaap"][s]["units"]["shares"]
                df_shares = pd.DataFrame(shares)
                duckdb.sql(f"""copy (
                    select '{t}' as ticker, 'shares' as item, * 
                    from df_shares) to '{path}/{t.lower()}_shares.parquet'
                """)
                break
            except Exception as e:
                continue

        try:
            eps = company_facts["facts"]["us-gaap"]["EarningsPerShareBasic"]["units"]["USD/shares"]
            df_eps = pd.DataFrame(eps)
            duckdb.sql(f"""copy (
                select '{t}' as ticker, 'eps' as item, *
                from df_eps) to '{path}/{t.lower()}_eps.parquet'
            """)
        except Exception as e:
            continue

### XLE

In [22]:
xle_catalog = duckdb.sql(f"""
    select
        etf, ticker, cik, name
        , industry, sic, fiscal_year
    from read_parquet('{PATH_HOLDINGS}/xle_holdings_metadata.parquet') 
""").df()

xle_catalog.head()

,etf,ticker,cik,name,industry,sic,fiscal_year
0,XLE,XOM,CIK0000034088,EXXON MOBIL CORP,Petroleum Refining,2911,1231
1,XLE,CVX,CIK0000093410,CHEVRON CORP,Petroleum Refining,2911,1231
2,XLE,COP,CIK0001163165,CONOCOPHILLIPS,Petroleum Refining,2911,1231
3,XLE,MPC,CIK0001510295,Marathon Petroleum Corp,Petroleum Refining,2911,1231
4,XLE,PSX,CIK0001534701,Phillips 66,Petroleum Refining,2911,1231


#### Verify concepts

In [23]:
xle_concepts_verified = duckdb.sql(f"""
    select tag, split_part(concept, '_', 2) as concept, count(*) as count
    from read_parquet('{PATH_HOLDINGS}/xle_concepts*', union_by_name=True)
    group by tag, concept
    order by tag, count desc
""").df()

# Proxies:
# OXY missing incomeloss --> stock used "us-gaap_ProfitLoss"
# Missing shares         --> get "us-gaap_EarningsPerShareBasic"

xle_concepts_verified

,tag,concept,count
0,incomeloss,NetIncomeLoss,20
1,revenue,RevenueFromContractWithCustomerExcludingAssess...,11
2,revenue,Revenues,7
3,revenue,RevenueFromContractWithCustomerIncludingAssess...,3
4,shares,WeightedAverageNumberOfSharesOutstandingBasic,16


In [24]:
xle_revenue_concepts = xle_concepts_verified[xle_concepts_verified.tag == 'revenue'].concept.tolist()

xle_incomeloss_concepts = xle_concepts_verified[xle_concepts_verified.tag == 'incomeloss'].concept.tolist()
xle_incomeloss_concepts.append('us-gaap_ProfitLoss')

xle_shares_concepts = xle_concepts_verified[xle_concepts_verified.tag == 'shares'].concept.tolist()

print(f'Revenue concepts {xle_revenue_concepts}')
print(f'Income concepts  {xle_incomeloss_concepts}')
print(f'Shares concepts  {xle_shares_concepts}')

Revenue concepts ['RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues', 'RevenueFromContractWithCustomerIncludingAssessedTax']
Income concepts  ['NetIncomeLoss', 'us-gaap_ProfitLoss']
Shares concepts  ['WeightedAverageNumberOfSharesOutstandingBasic']


#### Download data

In [25]:
# download_financials(
#     tickers=xle_catalog,
#     revenue_concepts=xle_revenue_concepts,
#     incomeloss_concepts=xle_incomeloss_concepts,
#     shares_concepts=xle_shares_concepts,
#     path=f'{PATH_FINANCIALS}/xle'
# )

## Download market data

In [26]:
def download_prices(tickers, path, interval='1d', period='max'):
    os.makedirs(path, exist_ok=True)
    unique_tickers = tickers.unique()
    print("Removed duplicate names.")
    unique_tickers = [ticker.replace('.', '-') for ticker in unique_tickers]
    print("Fixed ticker names.")

    for ticker in tqdm(unique_tickers):
        
        try:
            df = yf.download(ticker
                , interval=interval
                , period=period
                , multi_level_index=False
                , progress=False
                , auto_adjust=True
            ).reset_index()

            duckdb.sql(f"""
                copy (
                    select
                        '{ticker}' as ticker
                        , Date::date as date
                        , Open as open
                        , High as high
                        , Low as low
                        , Close as close
                        , Volume as volume
                        , (close*volume)::bigint as value
                    from df
                ) to '{path}/{ticker.lower()}_{interval}_prices.parquet'
            """)
            
        except Exception as e:
            print(f"{ticker} error: {e}")
            continue

### XLE

In [27]:
# download_prices(
#     tickers=xle_catalog.ticker,
#     path=f'{PATH_TECHNICALS}/xle',
#     interval='1d', period='max'
# )

# DATA PROCESSING

In [28]:
def process_fundamentals(df):
    """
    Every default 10-K gives values for the current year end, and last two comparable period.
    Every default 10-Q gives values for the current quarter end, and previous year's comparable quarter;
        also, the year-to-date values and its previous year comparable period.
    This means that the default contains a lot of redundant data.
    Fixed this by filtering with "where (date, period_end, days)", isolating the year-to-date values only.
        This makes it easier to compute quarterly data especially when dealing with 10-K.
        Also makes the flow more consistent instead of having separate code per form.
    It is therefore not a surprise, the dataframe shrinks by a lot after removing fixing the redundant reporting.
    Output: TTM numbers, with quarterly for reference. TTM makes it easier to get change as its always
        an annualized figure regardless of actual date and quarter of filing. 
        Better for trends than the typical quarter-over-quarter which is highly susceptible to cycles.
    """
    
    df = duckdb.sql(f"""
        with
        prep as (
            select
                ticker
                , item
                , form
                , try_cast(filed as date) as date
                , try_cast(start as date) as period_start
                , try_cast("end" as date) as period_end
                , (julian("end"::date) - julian("start"::date))::int as days
                , try_cast(val as numeric) as ytd
            from df
            where form in ('10-K', '10-Q')
        ),
        
        qtr as (
            select *
                , case when days < 100 then ytd
                    else ytd - lag(ytd, 1) over (partition by ticker order by period_end asc)
                    end as qtr
            from prep
            where (date, period_end, days) in (
                select date, max(period_end), max(days)
                from prep group by date
            )
        ),
        
        ttm as (
        select *
            , lag(qtr, 4) over (partition by ticker order by period_end asc) as prev_qtr
            , sum(qtr) over (partition by ticker order by period_end asc
                rows between 3 preceding and current row) as ttm
        from qtr
        where qtr is not null
        ),
        
        chg as (
        select *
            , try_cast(100 * ((qtr / prev_qtr) - 1) as numeric) as yoy_chg
            , try_cast(100 * ((ttm / lag(ttm, 1) over (partition by ticker order by period_end asc)) - 1) as numeric) as ttm_chg
        from ttm
        )
        
        select ticker, item, form, date, days, qtr, ttm, yoy_chg, ttm_chg
        from chg
        where yoy_chg is not null
        order by ticker asc, date desc
    """).df()

    return df

In [29]:
def process_shares(df):
    """
    Functions the same as process_fundamental, but skips compute for TTM, since
        share values are snapshots at end of period, much like balance sheet items.
    """
    df = duckdb.sql(f"""
        with
        days as (
            select 
                ticker
                , item
                , form
                , try_cast(filed as date) as date
                , try_cast("end" as date) as period_end
                , (julian("end"::date) - julian("start"::date))::int as days
                , try_cast(val as bigint) as shares
            from xle_shares
        )
    
        select * exclude (period_end, days)
        from days
        where (date, period_end, days) in (
            select date, max(period_end), max(days)
            from days
            group by date
            )
        order by ticker asc, date desc
    """).df()

    return df

## XLE

### Revenues

In [30]:
xle_revenues = duckdb.sql(f"""
    select * exclude (frame, accn)
    from read_parquet('{PATH_FINANCIALS}/xle/*revenues.parquet', union_by_name=True)
    where form in ('10-Q', '10-K')
    order by filed desc, "end" desc, val desc
""").df()

xle_revenues_processed = process_fundamentals(df=xle_revenues)
print(xle_revenues_processed.info())
# xle_revenues_processed.head()

<class 'pandas.DataFrame'>
RangeIndex: 287 entries, 0 to 286
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ticker   287 non-null    str           
 1   item     287 non-null    str           
 2   form     287 non-null    str           
 3   date     287 non-null    datetime64[us]
 4   days     287 non-null    int32         
 5   qtr      287 non-null    float64       
 6   ttm      287 non-null    float64       
 7   yoy_chg  287 non-null    float64       
 8   ttm_chg  287 non-null    float64       
dtypes: datetime64[us](1), float64(4), int32(1), str(3)
memory usage: 23.4 KB
None


In [31]:
xle_revenues_processed.groupby('ticker').item.count()

ticker
BKR     17
COP     19
CVX     18
DVN     16
EOG     39
EQT     17
FANG     3
HAL      1
KMI     20
MPC     17
OKE      2
OXY     19
PSX     18
SLB     17
TPL      8
TRGP    14
VLO     17
WMB     19
XOM      6
Name: item, dtype: int64

### Net income

In [32]:
xle_income = duckdb.sql(f"""
    select * exclude (frame, accn)
    from read_parquet('{PATH_FINANCIALS}/xle/*income.parquet', union_by_name=True)
    where form in ('10-Q', '10-K')
    order by filed desc, "end" desc, val desc
""").df()

xle_income_processed = process_fundamentals(df=xle_income)
xle_income_processed.info()
# xle_income_processed.head()

<class 'pandas.DataFrame'>
RangeIndex: 677 entries, 0 to 676
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ticker   677 non-null    str           
 1   item     677 non-null    str           
 2   form     677 non-null    str           
 3   date     677 non-null    datetime64[us]
 4   days     677 non-null    int32         
 5   qtr      677 non-null    float64       
 6   ttm      677 non-null    float64       
 7   yoy_chg  677 non-null    float64       
 8   ttm_chg  677 non-null    float64       
dtypes: datetime64[us](1), float64(4), int32(1), str(3)
memory usage: 53.7 KB


In [33]:
xle_income_processed.groupby('ticker').item.count()

ticker
BKR     19
COP     42
CVX     41
DVN     41
EOG     39
EQT     20
EXE     36
FANG    30
HAL     41
KMI     36
MPC     36
OKE     26
OXY     34
PSX     31
SLB     41
TPL      8
TRGP    34
VLO     41
WMB     40
XOM     41
Name: item, dtype: int64

### Earnings per share

In [34]:
xle_eps = duckdb.sql(f"""
    select * exclude (frame, accn)
    from read_parquet('{PATH_FINANCIALS}/xle/*eps.parquet', union_by_name=True)
    where form in ('10-Q', '10-K')
    order by filed desc, "end" desc, val desc
""").df()

xle_eps_processed = process_fundamentals(df=xle_eps)
xle_eps_processed.info()
# xle_eps_processed.head()

<class 'pandas.DataFrame'>
RangeIndex: 687 entries, 0 to 686
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ticker   687 non-null    str           
 1   item     687 non-null    str           
 2   form     687 non-null    str           
 3   date     687 non-null    datetime64[us]
 4   days     687 non-null    int32         
 5   qtr      687 non-null    float64       
 6   ttm      687 non-null    float64       
 7   yoy_chg  687 non-null    float64       
 8   ttm_chg  686 non-null    float64       
dtypes: datetime64[us](1), float64(4), int32(1), str(3)
memory usage: 52.5 KB


In [35]:
xle_eps_processed.groupby('ticker').item.count()

ticker
APA      8
BKR      3
COP     42
CVX     41
DVN     41
EOG     39
EQT     40
EXE     41
FANG    30
HAL     29
KMI     20
MPC     36
OKE     39
OXY     41
PSX     32
SLB     41
TPL      8
TRGP    34
VLO     41
WMB     40
XOM     41
Name: item, dtype: int64

### Shares

In [36]:
xle_shares = duckdb.sql(f"""
    select * exclude (frame, accn)
    from read_parquet('{PATH_FINANCIALS}/xle/*shares.parquet', union_by_name=True)
    where form in ('10-Q', '10-K')
    order by filed desc, "end" desc, val desc
""").df()

xle_shares_processed = process_shares(df=xle_shares)
xle_shares_processed.info()
# xle_shares_processed.head()

<class 'pandas.DataFrame'>
RangeIndex: 761 entries, 0 to 760
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   ticker  761 non-null    str           
 1   item    761 non-null    str           
 2   form    761 non-null    str           
 3   date    761 non-null    datetime64[us]
 4   shares  761 non-null    int64         
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 39.6 KB


In [37]:
xle_shares_processed.groupby('ticker').item.count()

ticker
APA     12
COP     47
CVX     45
DVN     32
EOG     44
EQT     44
EXE     45
FANG    35
HAL     45
KMI     25
MPC     39
OKE     43
OXY     44
PSX     37
SLB     45
TPL     12
TRGP    39
VLO     45
WMB     44
XOM     39
Name: item, dtype: int64

### Prices

In [38]:
xle_prices = duckdb.sql(f"""
    select *
    from read_parquet('{PATH_TECHNICALS}/xle/*.parquet', union_by_name=True)
""").df()

xle_prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 198523 entries, 0 to 198522
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   ticker  198523 non-null  str           
 1   date    198523 non-null  datetime64[us]
 2   open    198523 non-null  float64       
 3   high    198523 non-null  float64       
 4   low     198523 non-null  float64       
 5   close   198523 non-null  float64       
 6   volume  198523 non-null  int64         
 7   value   198523 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(2), str(1)
memory usage: 12.7 MB


### Joined data

In [39]:
xle_dataset = duckdb.sql(f"""
    with 
    clean as (
        select a.ticker, a.date::date as date, a.days
            , a.ttm as eps
            , b.ttm::bigint as income
            , d.ttm::bigint as revenue
            , shares::bigint as shares
        from      xle_eps_processed      a
        left join xle_income_processed   b on a.ticker=b.ticker and a.date=b.date
        left join xle_shares_processed   c on a.ticker=c.ticker and a.date=c.date
        left join xle_revenues_processed d on a.ticker=d.ticker and a.date=d.date
    ),
    derive as (
        select *
            , case when income is null then (eps*shares)::bigint else income end as income_derived
            , case when shares is null then (income/eps)::bigint else shares end as shares_derived
        from clean
    ),
    agg as (
        select a.ticker
            , a.date, days
            , shares_derived as shares
            , (shares_derived*close)::bigint as mcap
            , revenue
            , income_derived as income
            , (shares_derived*close/revenue)::numeric as ps
            , (shares_derived*close/income_derived)::numeric as pe
        from derive a
        left join xle_prices b on a.ticker=b.ticker and a.date=b.date
    ),
    lag as (
        select *
            , lead(mcap, 1) over (partition by ticker order by date asc) as mcap_lead_1
            , lead(mcap, 2) over (partition by ticker order by date asc) as mcap_lead_2
            , lead(mcap, 3) over (partition by ticker order by date asc) as mcap_lead_3
            , lag(pe, 1) over (partition by ticker order by date asc) as pe_lag_1
        from agg
    ),
    chg as (
        select * exclude (mcap_lead_1, mcap_lead_2, mcap_lead_3, pe_lag_1)
            , try_cast(((pe / pe_lag_1) - 1) * 100 as numeric) as pe_chg
            , try_cast(((mcap_lead_1 / mcap) - 1) * 100 as numeric) as mcap_fwd_chg_1
            , try_cast(((mcap_lead_2 / mcap) - 1) * 100 as numeric) as mcap_fwd_chg_2
            , try_cast(((mcap_lead_3 / mcap) - 1) * 100 as numeric) as mcap_fwd_chg_3
        from lag
    )
    select *
    from chg
    where pe_chg is not null    
    order by ticker asc, date desc
""").df()

In [40]:
xle_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 636 entries, 0 to 635
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ticker          636 non-null    str           
 1   date            636 non-null    datetime64[us]
 2   days            636 non-null    int32         
 3   shares          636 non-null    int64         
 4   mcap            636 non-null    int64         
 5   revenue         266 non-null    Int64         
 6   income          636 non-null    int64         
 7   ps              266 non-null    float64       
 8   pe              636 non-null    float64       
 9   pe_chg          636 non-null    float64       
 10  mcap_fwd_chg_1  614 non-null    float64       
 11  mcap_fwd_chg_2  593 non-null    float64       
 12  mcap_fwd_chg_3  573 non-null    float64       
dtypes: Int64(1), datetime64[us](1), float64(6), int32(1), int64(3), str(1)
memory usage: 64.8 KB


In [41]:
xle_dataset.head()

,ticker,date,days,shares,mcap,revenue,income,ps,pe,pe_chg,mcap_fwd_chg_1,mcap_fwd_chg_2,mcap_fwd_chg_3
0,APA,2026-05-07,89,354000000,12828960594,<NA>,1097400000,NaN,11.690,36.327,NaN,NaN,NaN
1,APA,2025-02-28,365,353000000,6901818985,<NA>,804840000,NaN,8.575,195.384,85.878,NaN,NaN
2,APA,2024-11-07,273,348000000,7132320694,<NA>,2456880000,NaN,2.903,1.503,-3.232,79.871,NaN
3,APA,2024-08-02,181,337000000,8914796970,<NA>,3117250000,NaN,2.860,-4.762,-19.995,-22.580,43.906
4,APA,2024-05-02,90,302000000,8090110245,<NA>,2693840000,NaN,3.003,1.009,10.194,-11.839,-14.688


In [44]:
print(xle_dataset[xle_dataset.pe_chg > 0][['pe_chg', 'mcap_fwd_chg_1', 'mcap_fwd_chg_2', 'mcap_fwd_chg_3']].describe())

              pe_chg  mcap_fwd_chg_1  mcap_fwd_chg_2  mcap_fwd_chg_3
count     324.000000      306.000000      294.000000      278.000000
mean     1037.187259      662.942467      611.930626      664.066640
std     10298.524462     8068.478778     7259.390721    10582.534703
min         0.038000      -99.895000      -99.892000      -99.853000
25%         9.657500       -7.165500      -10.787750      -11.281750
50%        28.023500        3.708000        8.276500       11.176500
75%        67.746750       15.493500       35.067750       48.077000
max    143810.000000   106132.115000   100708.250000   176471.197000


In [43]:
print(xle_dataset[xle_dataset.pe_chg < 0][['pe_chg', 'mcap_fwd_chg_1', 'mcap_fwd_chg_2', 'mcap_fwd_chg_3']].describe())

              pe_chg  mcap_fwd_chg_1  mcap_fwd_chg_2  mcap_fwd_chg_3
count     309.000000      305.000000      296.000000      292.000000
mean     -774.795683     1140.755849     1327.116926     1641.867449
std      8782.049252    15752.775709    14895.535032    20972.065708
min   -149687.500000      -99.832000      -65.098000      -99.891000
25%       -80.806000       -9.169000      -12.609000      -16.951250
50%       -31.113000        0.338000        1.644000        5.264000
75%       -10.739000       14.598000       21.599000       33.699250
max        -0.154000   262498.675000   231091.104000   347903.865000
